## Configure API KEYS

In [1]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter GOOGLE_API_KEY: ")

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter TAVILY_API_KEY: ")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")

print("GOOGLE_API_KEY configured:", bool(os.getenv("GOOGLE_API_KEY")))
print("GROQ_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))

# print("GOOGLE_API_KEY configured:", os.getenv("GOOGLE_API_KEY"))
# print("GROQ_API_KEY configured:", os.getenv("GROQ_API_KEY"))
# print("TAVILY_API_KEY configured:", os.getenv("TAVILY_API_KEY"))
# print("PINECONE_API_KEY configured:", os.getenv("PINECONE_API_KEY"))

GOOGLE_API_KEY configured: True
GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


### Load the Document 

In [8]:
from langchain_community.document_loaders import WebBaseLoader

source_url = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_path=source_url,
    requests_kwargs = {
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    }
)
raw_docs = loader.load()

print(f"Loaded document: {len(raw_docs)}\n documents from: {source_url}")
print("Source: ", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1000])

Loaded document: 1
 documents from: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Source:  https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase stu

### Split the document into Chunk

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    add_start_index = True,
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Total chunks: 27

First chunk preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase studiesGet helpOn this pageConceptsSetupSet up LangSmithPreprocess documentsCre


### Create Embeddings Using Gemini Model

In [24]:
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

gemini_result = gemini_client.models.embed_content(
    model="gemini-embedding-2",
    contents="What is agentic RAG in LangChain and how does it work?",
    config = types.EmbedContentConfig(
        output_dimensionality=768
    )
)
print("Gemini Embedding Result: ", gemini_result.embeddings)

[embedding_obj] = gemini_result.embeddings
embedding_length = len(embedding_obj.values)
print("Gemini Embedding Result Length: ", embedding_length)

Gemini Embedding Result:  [ContentEmbedding(
  values=[
    0.01283944,
    0.013940322,
    0.02108872,
    0.027043901,
    -0.007166158,
    <... 763 more items ...>,
  ]
)]
Gemini Embedding Result Length:  768
